# RBI Data Quality Pipeline: Stage 2 (Quality Checks)

Stage one cleaned the RBI's weekly Foreign Exchange Reserves table into
1,334 weekly observations and found some data quality problems by hand.
This notebook turns the first of those checks into code that can run on
any batch of data.

The first check looks at missing values: how many each column has, and
the longest unbroken run of missing weeks. I already know the answers
from stage one (for example, 52 weeks in a row with no IMF values), so
I can use them to test that the check works.

Later stages will add more checks, such as outliers and sudden changes.

In [1]:
import pandas as pd
data = pd.read_csv("table2_clean.csv",parse_dates=["week_ended"])
print(data.shape)
print(data.dtypes)

(1334, 11)
week_ended            datetime64[ns]
total_reserves_inr           float64
total_reserves_usd           float64
fca_inr                      float64
fca_usd                      float64
gold_inr                     float64
gold_usd                     float64
sdr_inr                      float64
sdr_usd                      float64
imf_inr                      float64
imf_usd                      float64
dtype: object


In [2]:
print(data.isna().sum())

week_ended             0
total_reserves_inr     0
total_reserves_usd     0
fca_inr                0
fca_usd                0
gold_inr               0
gold_usd               0
sdr_inr                0
sdr_usd               16
imf_inr               52
imf_usd               52
dtype: int64


## First attempt: longest gap in one column

Before writing a general function, I tried the idea on one column where I
already know the answer, `imf_inr`. The loop walks down the weeks with two
counters: how long the current gap is, and the longest gap seen so far.
It finds 52, which matches the block I found by hand in stage one.

In [3]:
current_run = 0
longest_run = 0
for is_missing in data["imf_inr"].isna():
    if is_missing == True:
        current_run = current_run+1
        if current_run > longest_run:
            longest_run = current_run
    else:
        current_run = 0
print(longest_run)
        

52


## Turning it into a function

To check every column, I wrapped the loop in a function. It also remembers
where each gap starts, so it can report the dates of the longest one. It
returns the length of the longest gap and the rows where it starts and ends.
If there are no gaps, the start and end are `None`. If two gaps tie, the
first one is kept.

In [4]:
def longest_missing_run(column_name):
    current_run = 0
    longest_run = 0
    current_start = None
    longest_start = None
    longest_end = None
    for i, is_missing in enumerate(column_name.isna()):
        if is_missing == True:
            current_run = current_run+1
            if current_run == 1:
                current_start = i
            if current_run > longest_run:
                longest_run = current_run
                longest_start = current_start
                longest_end = i
        else:
            current_run = 0
     
   
    return longest_run, longest_start, longest_end


## The report

For each number column, the report prints the missing count, the longest gap,
where it sits, and a verdict. A column is flagged when its longest gap is
more than 4 weeks, about a month of weekly data. The 4 is a judgment call,
and I plan to test it later.

In [5]:
max_allowed_gap = 4
for columns in data.columns[1:]:
    gap,start,end = longest_missing_run(data[columns])

    if start is None:
        date = "No Gap"
    else:
        d1 = data["week_ended"].iloc[start].date() 
        d2 = data["week_ended"].iloc[end].date()
        date = f"{d1} to {d2}"
    if gap > max_allowed_gap:
        verdict = "FLAGGED"
    else:
        verdict = "OK"
    print(columns,data[columns].isna().sum(),gap,date,verdict)

total_reserves_inr 0 0 No Gap OK
total_reserves_usd 0 0 No Gap OK
fca_inr 0 0 No Gap OK
fca_usd 0 0 No Gap OK
gold_inr 0 0 No Gap OK
gold_usd 0 0 No Gap OK
sdr_inr 0 0 No Gap OK
sdr_usd 16 6 2006-06-09 to 2006-07-14 FLAGGED
imf_inr 52 52 2001-04-06 to 2002-03-29 FLAGGED
imf_usd 52 52 2001-04-06 to 2002-03-29 FLAGGED


The three flagged columns are the ones I found by hand in stage one. One
limit: the check only measures gap length, so it treats the `sdr_usd` blanks
(probably rounding of tiny amounts) the same as the IMF gap (probably a
reporting gap).

In [6]:
print(longest_missing_run(pd.Series([1, None, None])))
print(longest_missing_run(pd.Series([None, None, None])))
print(longest_missing_run(pd.Series([1, 2, 3])))
print(longest_missing_run(pd.Series([None, 1, None, None, None, None])))

(2, 1, 2)
(3, 0, 2)
(0, None, None)
(4, 2, 5)
